<p style="font-size:16pt; font-weight:bold; color:red; padding-bottom:20px; float:right">Please rename this file before editing!</p>

# Session 8 — Thinking in Tables

Session 7 gave you the tools: NumPy arrays, pandas tables, joins, groups. This
session is about the **habits** that make those tools produce answers you can
trust.

Five ideas, one part each:

| Part | Idea | In one sentence |
|---|---|---|
| 1 | The table is the central structure | Rows are observations, columns are variables, and that is all you need to remember about the layout. |
| 2 | Vectorized thinking | Say what happens to a whole column, not what happens to each row. |
| 3 | Split, apply, combine | Every "per something" question has the same three steps. |
| 4 | A missing value is a question | A blank can mean zero, unknown, or not applicable, and the three give different numbers. |
| 5 | Inspect before you conclude | Five lines of looking, before the one line of computing. |

**How this session's notebooks fit together**

| Notebook | What it is |
|---|---|
| `01_Thinking_in_Tables` (this one) | The ideas, on tables small enough to read |
| `02_Yelp_Businesses_Example` | One real analysis from raw file to conclusion, using all five |
| `03_Data_Analysis_Reference` | A lookup guide to keep open while you do Homework 7 |

**What is not repeated here.** Session 7 covered these, and this session uses
them without explaining them again:

| Topic | Where |
|---|---|
| Series and DataFrame, choosing rows and columns, sorting | `07-Data-Tables/01_Intro_Numpy_Pandas` |
| Grain, joins, grouping basics, ranking within a group | `07-Data-Tables/02_Northwind_SQL_Pandas` |
| `.loc` and `.iloc`, alignment, reshaping, dates and resampling | `07-Data-Tables/03_Array_DataFrame_Operations` |

**Before you start**

- Set the kernel (upper right corner) to **Conda Python 3.12**.
- Run the cells in order with **Shift + Enter**. Before each cell, guess what
  it will show.

In [ ]:
import time

import numpy as np
import pandas as pd

pd.options.display.float_format = "{:,.2f}".format

---

## Part 1 — The table is the central structure

Since Session 3 you have kept records as a **list of dictionaries**. Here is a
week of deliveries from a small farm to four restaurants:

In [ ]:
deliveries = [
    {"day": "Mon", "customer": "Bistro",  "product": "eggs",    "boxes": 4, "price": 12.0},
    {"day": "Mon", "customer": "Cafe",    "product": "berries", "boxes": 2, "price": 30.0},
    {"day": "Tue", "customer": "Bistro",  "product": "berries", "boxes": 1, "price": 30.0},
    {"day": "Tue", "customer": "Diner",   "product": "eggs",    "boxes": 6, "price": 12.0},
    {"day": "Wed", "customer": "Cafe",    "product": "greens",  "boxes": 3, "price": 18.0},
    {"day": "Thu", "customer": "Diner",   "product": "eggs",    "boxes": 5, "price": 12.0},
    {"day": "Thu", "customer": "Bistro",  "product": "greens",  "boxes": 2, "price": 18.0},
    {"day": "Fri", "customer": "Tavern",  "product": "berries", "boxes": 4, "price": 30.0},
]
len(deliveries)

**The question:** how much did each customer spend this week?

The way you have done it since Homework 2 — a loop, and a dictionary whose keys
come from the data:

In [ ]:
spent = {}
for record in deliveries:
    amount = record["boxes"] * record["price"]
    spent[record["customer"]] = spent.get(record["customer"], 0) + amount
spent

Now the same records as a **table**. One call does the conversion, because a
list of dictionaries already has the right shape: each dictionary is a row,
each key is a column.

In [ ]:
table = pd.DataFrame(deliveries)
table

In [ ]:
table["amount"] = table["boxes"] * table["price"]
table.groupby("customer")["amount"].sum()

Same answer, two lines, no loop and no dictionary to manage. The difference is
not the typing saved. It is what you had to think about:

| With the loop you decide... | With the table you decide... |
|---|---|
| how to walk through the records | *(nothing: pandas walks)* |
| where to keep the running totals | *(nothing: pandas keeps them)* |
| what to do when a key is new | *(nothing)* |
| **what to compute, and per what** | **what to compute, and per what** |

Only the last row is the actual question.

### Rows are observations, columns are variables

That sentence is the whole layout rule.

- A **row** is one thing that happened or one thing that exists: one delivery.
- A **column** is one fact recorded about every such thing: its day, its
  customer, how many boxes.

When a table follows the rule, every question becomes "do something to a
column" or "do something per group of rows". When it does not, nothing works
until you reshape it. (Session 7's third notebook shows how, with `melt` and
`pivot`.)

Before using any table, say out loud what one row is. That is the table's
**grain**, and Homework 6 was about what happens when you get it wrong.

**✏️ Try it:** Total boxes per **product**, in one line. Then: which column would you group by to get the busiest **day**?

---

## Part 2 — Vectorized thinking

`table["boxes"] * table["price"]` multiplied eight pairs of numbers, and you
never wrote which pair. That is a **vectorized** operation: you describe what
happens to the **whole column**, and pandas applies it to every row.

### The same formula, two ways

In [ ]:
# Row by row -- what you would write in core Python.
amounts = []
for index, row in table.iterrows():
    amounts.append(row["boxes"] * row["price"])
amounts

In [ ]:
# The whole column at once.
(table["boxes"] * table["price"]).tolist()

On eight rows nobody can tell the difference. Make it a million.

In [ ]:
rng = np.random.default_rng(8410)
big = pd.DataFrame({
    "boxes": rng.integers(1, 10, size=1_000_000),
    "price": rng.choice([12.0, 18.0, 30.0], size=1_000_000),
})

start = time.perf_counter()
fast = big["boxes"] * big["price"]
vector_seconds = time.perf_counter() - start
print(f"whole column, 1,000,000 rows: {vector_seconds:.4f} s")

In [ ]:
# The loop, on only 50,000 of the rows -- the full million would take a while.
part = big.head(50_000)

start = time.perf_counter()
slow = []
for index, row in part.iterrows():
    slow.append(row["boxes"] * row["price"])
loop_seconds = time.perf_counter() - start

print(f"row by row, 50,000 rows: {loop_seconds:.2f} s")
print(f"so a million rows would take about {loop_seconds * 20:.0f} s")
print(f"the whole-column version is roughly "
      f"{loop_seconds * 20 / vector_seconds:,.0f} times faster")

**Why so different?** The loop still happens in the fast version. It happens
inside NumPy, in compiled code, on a block of numbers that all have the same
type. The Python loop instead builds a small object for every row, looks each
column up by name, and checks the type of every value before multiplying,
a million times over.

Speed is the smaller benefit. The larger one is that the vectorized line
**reads like the formula**: amount = boxes × price.

### Conditions are columns too

A comparison on a column gives a column of `True` and `False`:

In [ ]:
table["boxes"] >= 4

That column can do three jobs.

In [ ]:
big_order = table["boxes"] >= 4

print("1. choose rows:")
print(table[big_order][["day", "customer", "boxes"]])

print("\n2. count them:  ", big_order.sum())       # True counts as 1
print("3. their share:  ", big_order.mean())       # the mean of True/False

Remember number 3. **The mean of a True/False column is a share.** It turns
"what fraction of the rows..." into one line, and you will use it constantly.

### A value that depends on a condition

In a loop you would write `if ... else`. For a whole column, `np.where`:

In [ ]:
table["size"] = np.where(table["boxes"] >= 4, "large", "small")
table[["customer", "boxes", "size"]]

### The habit

> When you are about to write `for` over the rows of a table, stop and ask:
> *what happens to the column?*

A loop over a **handful of things** — three file names, four policies — is
still fine. A loop over the **rows of a table** almost never is.

**✏️ Try it:** Add a column `discounted` that is `amount * 0.9` where `boxes >= 4` and plain `amount` otherwise. No loop.

---

## Part 3 — Split, apply, combine

"Per customer", "per day", "per product": every question with a *per* in it
has the same three steps.

```text
        SPLIT                    APPLY                  COMBINE
 the rows, by a key      a summary to each group     the results

 Bistro | eggs    48 ┐
 Bistro | berries 30 ├──▶  sum  ──▶  114 ┐
 Bistro | greens  36 ┘                   │      customer  amount
                                         │      Bistro       114
 Cafe   | berries 60 ┐                   ├──▶   Cafe         114
 Cafe   | greens  54 ┴──▶  sum  ──▶  114 │      Diner        132
                                         │      Tavern       120
 Diner  | eggs    72 ┐                   │
 Diner  | eggs    60 ┴──▶  sum  ──▶  132 │
                                         │
 Tavern | berries 120 ──▶  sum  ──▶  120 ┘
```

### By hand first

Do the three steps yourself once, so `groupby` is not magic.

In [ ]:
results = {}
for customer in table["customer"].unique():
    group = table[table["customer"] == customer]        # SPLIT
    results[customer] = group["amount"].sum()           # APPLY
pd.Series(results, name="amount")                       # COMBINE

(That loop is over four customers, not over the rows. It is the allowed kind.)

### With `groupby`

In [ ]:
table.groupby("customer")["amount"].sum()

`groupby("customer")` is the split. `["amount"]` says which column to
summarise. `.sum()` is the apply, and pandas combines.

You can look inside the split:

In [ ]:
for customer, group in table.groupby("customer"):
    print(customer, "-", len(group), "deliveries")
    print(group[["day", "product", "amount"]].to_string(index=False))
    print()

### Three things to do with a group

The *apply* step comes in three kinds, and choosing the right one is most of
the skill.

| Kind | Each group becomes | The result has | Use it for |
|---|---|---|---|
| **aggregate** — `agg` | one value | one row per group | totals, counts, averages |
| **transform** — `transform` | one value **per row** | the same rows as before | "compared with its group" |
| **filter** — `filter` | kept or dropped, whole | some of the original rows | "only the groups that..." |

In [ ]:
# AGGREGATE: several summaries at once.  new_column=(column_to_use, how)
table.groupby("customer", as_index=False).agg(
    deliveries=("amount", "size"),
    boxes=("boxes", "sum"),
    amount=("amount", "sum"),
)

In [ ]:
# TRANSFORM: the group's total, written next to every one of its rows.
table["customer_total"] = table.groupby("customer")["amount"].transform("sum")
table["share"] = table["amount"] / table["customer_total"]
table[["customer", "product", "amount", "customer_total", "share"]]

In [ ]:
# FILTER: keep only the customers with more than two deliveries.
table.groupby("customer").filter(lambda group: len(group) > 2)[
    ["day", "customer", "product", "amount"]]

Notice the shapes. `agg` gave four rows, one per customer. `transform` gave
eight, one per delivery. `filter` gave three, the deliveries of the one
customer that qualified.

> **`agg` shrinks the table. `transform` keeps its shape.**

If you need a group's number *beside each row* — a share, a difference from
the group average, a value to fill a blank with — you want `transform`.

**✏️ Try it:** For each delivery, how far is its `amount` above or below the **average amount of the same product**? (`transform("mean")`, then subtract.)

---

## Part 4 — A missing value is a question

The farm starts recording tips. The driver fills in the column when they
remember to.

In [ ]:
table["tip"] = [5.0, np.nan, 0.0, np.nan, 4.0, np.nan, 3.0, np.nan]
table[["day", "customer", "amount", "tip"]]

`NaN` is how pandas writes "no value here". Four of eight tips are missing.
**What was the average tip?**

In [ ]:
print("leave the blanks out: ", table["tip"].mean())
print("count them as zero:   ", table["tip"].fillna(0).mean())

3.00 or 1.50. One is double the other, and pandas computed both without a
word of complaint. Which is right depends on what a blank **means**, and that
is not in the data. It is in how the data was collected.

| A blank means | Example | Then you should |
|---|---|---|
| **Zero** — nothing happened | no tip was given, so nothing was written | fill with 0 |
| **Unknown** — something happened, nobody recorded it | the driver forgot to write the tip down | leave it out, and say how many you left out |
| **Not applicable** — the question does not fit | the customer pays by invoice; there is no tip | leave it out; it was never part of the average |

Notice that one row already says `0.0`. Somebody **did** write a zero there.
So in this table a zero and a blank are different things, which is a hint that
the blanks are "unknown" and not "zero". Hints like that are all you usually
get.

### How missing values behave

Four rules. They are consistent, and each one can surprise you.

In [ ]:
tip = table["tip"]
print("1. sum, mean, min, max SKIP missing values:   sum =", tip.sum())
print("2. count() counts only the values present:    count =", tip.count(),
      " but len =", len(tip))
print("3. arithmetic with a missing value is missing:")
print((table["amount"] + tip).tolist())
print("4. a comparison with a missing value is False:")
print((tip > 2).tolist())

Rule 2 is why `mean()` gave 3.00: it divided by 4, not by 8.

Rule 4 hides a trap. `tip > 2` is `False` for the blanks, **and so is**
`tip <= 2`. The missing rows are in neither half. If you split a table with a
condition and its opposite, rows can silently fall through the middle.

In [ ]:
print("tip > 2:   ", (tip > 2).sum(), "rows")
print("tip <= 2:  ", (tip <= 2).sum(), "rows")
print("neither:   ", tip.isna().sum(), "rows   <- easy to lose")

### A fifth rule, for groups

`groupby` **drops rows whose key is missing**, unless you say otherwise.

In [ ]:
table["region"] = ["north", "north", None, "south", "south", None, "north", "south"]

print(table.groupby("region")["amount"].sum())
print("\nsum of the groups:", table.groupby("region")["amount"].sum().sum())
print("sum of the table: ", table["amount"].sum())

In [ ]:
table.groupby("region", dropna=False)["amount"].sum()

90 of the 480 in revenue vanished from the first version, without an error.
`dropna=False` keeps the unknowns as a group of their own, so the totals add
up. You met this in Homework 6.

### Missing values in disguise

`isna()` finds `NaN`. It does not find a blank that somebody typed something
into.

In [ ]:
survey = pd.DataFrame({
    "customer": ["Bistro", "Cafe", "Diner", "Tavern", "Grill"],
    "rating":   [4, -999, 5, 3, -999],
    "comment":  ["great eggs", "N/A", "", "fine", "unknown"],
})
print(survey)
print("\nmissing, says isna():")
print(survey.isna().sum())
print("\naverage rating:", survey["rating"].mean())

Nothing is missing, says pandas, and the average rating is −397.

`-999`, `"N/A"`, `""` and `"unknown"` are all blanks wearing a costume. You
find them by **looking** — which is Part 5 — and then you turn them into real
missing values:

In [ ]:
cleaned = survey.copy()
cleaned["rating"] = cleaned["rating"].where(cleaned["rating"] >= 0)
words = cleaned["comment"].str.strip().str.lower()
cleaned["comment"] = cleaned["comment"].mask(words.isin(["n/a", "", "unknown"]))

print(cleaned)
print("\naverage rating:", cleaned["rating"].mean(),
      "from", cleaned["rating"].count(), "of", len(cleaned), "answers")

`where(condition)` keeps a value where the condition is true and blanks it
everywhere else. `mask(condition)` is the opposite. Those two methods do most
of the cleaning in Homework 7.

**The honest way to report a number with missing data behind it** is the last
line above: the number, **and how many rows it came from**.

**✏️ Try it:** What is the average tip **per customer**, leaving blanks out? Which customer has no average at all, and why is that the right answer?

---

## Part 5 — Inspect before you conclude

Every problem in Part 4 was visible before any computing was done. Nobody
looked.

A table arrives. Before the first question, five lines:

| | Line | It answers |
|---:|---|---|
| 1 | `table.shape` | How many rows and columns? Is that what I expected? |
| 2 | `table.dtypes` | Is every number a number and every date a date? |
| 3 | `table.head()` / `table.sample(5)` | What does a row actually look like? |
| 4 | `table.isna().sum()` | Where are the blanks, and how many? |
| 5 | `table.describe()` / `value_counts()` | Are the smallest, largest and most common values believable? |

Here is a small table with four things wrong with it. Try to find them with
the five lines before reading on.

In [ ]:
orders = pd.DataFrame({
    "order":    [101, 102, 103, 104, 105, 106, 107, 108],
    "customer": ["Bistro", "bistro", "Cafe", "Diner", "Cafe ", "Diner", "Tavern", "Bistro"],
    "boxes":    [4, 2, 3, 9999, 1, 6, 5, 2],
    "price":    ["12.00", "30.00", "18.00", "12.00", "$30.00", "12.00", "30.00", "18.00"],
    "paid":     ["yes", "yes", "no", "yes", None, "yes", "no", "yes"],
})
orders

In [ ]:
print(orders.shape)
print()
print(orders.dtypes)

**Problem 1.** `price` is `object`, which means text. One `$` sign in one cell
was enough: pandas could not read that cell as a number, so it kept the whole
column as text. Try to total it and see what text does:

In [ ]:
orders["price"].sum()

No error. `+` on text glues it together. A wrong type rarely fails. It gives
you nonsense quietly.

In [ ]:
print(orders.isna().sum())
print()
print(orders.describe())

**Problem 2.** One `paid` is missing. Is an order with no entry unpaid, or
unknown? (Part 4.)

**Problem 3.** The largest `boxes` is 9,999 and the mean is about 1,253, on a farm
that delivers a handful of boxes at a time. That is a placeholder, a blank in
disguise, and it is carrying the whole average.

In [ ]:
orders["customer"].value_counts()

**Problem 4.** Six customers, where there are four. `Bistro` and `bistro` are
different text, and so are `Cafe` and `Cafe ` with a space after it. Any
`groupby("customer")` would have split them.

Now fix what you found, and the question can be asked:

In [ ]:
fixed = orders.copy()
fixed["price"] = pd.to_numeric(fixed["price"].str.replace("$", "", regex=False))
fixed["boxes"] = fixed["boxes"].where(fixed["boxes"] < 1000)
fixed["customer"] = fixed["customer"].str.strip().str.title()
fixed["amount"] = fixed["boxes"] * fixed["price"]

fixed.groupby("customer", as_index=False).agg(
    orders=("order", "size"),
    amount=("amount", "sum"),
    orders_with_amount=("amount", "count"),
)

Look at the last column. Diner has two orders and an amount from only one of
them, because the 9,999 became a blank and a blank times a price is a blank.
The table tells you so, **because you asked it to**.

### The order of work

```text
 inspect  ──▶  clean  ──▶  compute  ──▶  inspect the result
    ▲                                          │
    └──────────────────────────────────────────┘
```

The arrow back is not decoration. A result is a table too. Does it have the
number of rows you expected? Do the parts add up to the whole?

---

## Exercises

Use `table` (the deliveries, with the columns added above) and `fixed`.

### 🧩 Exercise 1 — Vectorize

Add a column `heavy` to `table` that is `True` when a delivery is eggs **and**
has at least 5 boxes. Then give, in one line each, the number of heavy
deliveries and their share of all deliveries.

Remember that each comparison needs its own parentheses when you combine them
with `&`.

### 🧩 Exercise 2 — Split, apply, combine

One table with one row per **product**: number of deliveries, total boxes,
total amount, and the number of *different* customers who bought it. Sort it
by amount, largest first.

### 🧩 Exercise 3 — Aggregate or transform?

For every delivery, compute its amount as a share of **that day's** total
amount. Which of `agg` and `transform` do you need, and how can you tell
before writing any code?

### 🧩 Exercise 4 — What does the blank mean?

Compute the average tip three ways: leaving blanks out, counting them as zero,
and filling each blank with the average tip of the **same customer**. For the
third, which blanks could not be filled, and what did you do with them?

Write one sentence saying which of the three you would report, and what you
would write beside it.

---

## Summary

| Idea | The habit | The tool |
|---|---|---|
| The table is the central structure | Say what one row is before you use it | `pd.DataFrame(list_of_dicts)` |
| Vectorized thinking | Describe the column, not the row | arithmetic and comparisons on columns, `np.where` |
| Split, apply, combine | Hear the *per* in the question | `groupby` with `agg`, `transform` or `filter` |
| A missing value is a question | Decide what a blank means, then say so | `isna`, `fillna`, `where`, `mask`, `dropna=False` |
| Inspect before you conclude | Five lines of looking first | `shape`, `dtypes`, `head`, `isna().sum()`, `describe` |

### Where each idea appears in Homework 7

| Idea | Homework 7 |
|---|---|
| Inspect before you conclude | Phase 1, `inspect` |
| Missing values in disguise, `where` and `mask` | Phase 2, `clean` |
| Vectorized thinking, `np.where` and its cousin `np.select` | Phase 3, `add_infested_acres`, `add_size_class` |
| A missing value is a question | Phase 4, `fill_cover`, `compare_policies` |
| Split, apply, combine — aggregate | Phase 5, `by_forest` |
| Split, apply, combine — transform | Phase 6, `add_share_of_forest` |

**Next:** `02_Yelp_Businesses_Example` runs all five ideas on 150,000 real
records, and `03_Data_Analysis_Reference` is the lookup guide.